## Implementación
En este notebook se pueden encontrar:
* Análisis exploratorio de datos
* Bloque para procesamiento de datos (feaature engineering)
* Bloque para validación de modelos
* Bloque para búsqueda de hiperparámetros
* Modelo final y exportación de predicciones

### Análisis exploratorio de datos

In [2]:
# Importamos librerías y funciones necesarias
import pandas as pd
import numpy as np

In [3]:
# Cargamos los datos de entrenamiento y de submission
data = pd.read_csv('../competition_data.csv')
submission = pd.read_csv('../submission.csv')

### Feature engineering
En esta sección mostramos como se utilizan los métodos implementados para procesar los datos y obtener todos los atributos creados. La implementación de los métodos se encuentra en el archivo funciones.py

In [4]:
import importlib
import funciones as func # Tener funciones.py en mismo directorio. Tiene las funciones usadas para procesar un df
importlib.reload(func)

<module 'funciones' from 'c:\\Users\\dafyd\\Documents\\Escuela\\2025\\semestre 1\\TD6\\TP2\\entregar\\funciones.py'>

#### Utilización de la API de Spotify para obtener duración de canciones

In [5]:
# Necesitamos obtener las duraciones de los tracks de ambos conjuntos de datos
# uri_to_ms_data = func.get_songs_durations(data)
# uri_to_ms_submission = func.get_songs_durations(submission)
# diccionario = {**uri_to_ms_data, **uri_to_ms_submission}

In [6]:
import csv
# Guardamos el diccionario en un archivo CSV para no tener que volver a calcularlo
# f = open('uri_to_duration.csv', 'w')
# f.write('uri,duration\n')
# for key, value in diccionario.items():
#     f.write(f"{key},{value}\n")
# f.close()

In [7]:
f = open('uri_to_duration.csv', 'r')
diccionario = {}
for line in csv.DictReader(f):
    diccionario[line['uri']] = float(line['duration'])
f.close()

#### Cálculo de todos los atributos agregados para data y submission

In [8]:
# Procesamiento de los datos de entrenamiento
data = func.procesar_df(data, diccionario) # Esta función deja a dara con todos los atributos que usamos en el modelo
data

,Unnamed: 0,ts,platform,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,reason_start,shuffle,TARGET,...,reason_fwdbtn,reason_playbtn,reason_remote,reason_trackdone,reason_trackerror,reason_unknown,track_prop,artist_prop,album_prop,hour day_of_week
0,74917,2014-06-27 18:01:16+00:00,"iOS 7.0.4 (iPod5,1)",Algo Demencial,Los Tipitos,Push,spotify:track:0rqD0zvqI4GrvyUxyQ7Ij3,fwdbtn,True,True,...,True,False,False,False,False,False,0.000010,0.002067,0.000679,72.0
1,74918,2014-06-29 20:10:10+00:00,"iOS 7.0.4 (iPod5,1)",Iron Man - Live,Black Sabbath,Past Lives,spotify:track:1XtZ3GROvmy4JrT6uMvsD8,NaN,False,True,...,False,False,False,False,False,False,0.000020,0.000429,0.000020,120.0
2,74919,2014-06-29 20:10:54+00:00,"iOS 7.0.4 (iPod5,1)",Sultans Of Swing,Dire Straits,Dire Straits,spotify:track:3LTMnFa0hhwisyq6ILahyj,fwdbtn,False,True,...,True,False,False,False,False,False,0.000389,0.000869,0.000310,120.0
3,74920,2014-06-29 20:11:16+00:00,"iOS 7.0.4 (iPod5,1)",Fortunate Son,Creedence Clearwater Revival,Willy And The Poor Boys,spotify:track:7I2lPuuiOkpKtZlhr4zUpT,fwdbtn,False,True,...,True,False,False,False,False,False,0.000399,0.001238,0.000669,120.0
4,74921,2014-09-04 21:46:24+00:00,"iOS 7.0.4 (iPod5,1)","Comptine d'un autre été, l'après-midi",Yann Tiersen,Amelie from Montmartre,spotify:track:2AkcjsKlRbIBYGAgpQVFii,NaN,False,True,...,False,False,False,False,False,False,0.000020,0.000020,0.000020,63.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100139,74911,2024-05-23 17:24:56+00:00,ios,The Tide Is High,Blondie,Atomic/Atomix,spotify:track:52Rp3xBJFYYdmpgzDy0Quf,trackdone,True,False,...,False,False,False,True,False,False,0.000409,0.000759,0.000419,51.0
100140,74912,2024-05-23 17:28:33+00:00,ios,The Less I Know The Better,Tame Impala,Currents,spotify:track:6K4t31amVTZDgR3sKmwUJJ,trackdone,True,False,...,False,False,False,True,False,False,0.001907,0.002277,0.002087,51.0
100141,74913,2024-05-23 23:44:01+00:00,ios,Pink + White,Frank Ocean,Blonde,spotify:track:3xKsf9qdS1CyvXSMEid6g8,trackdone,True,False,...,False,False,False,True,False,False,0.000040,0.000260,0.000050,69.0
100142,74914,2024-05-23 23:50:25+00:00,ios,Still Got The Blues,Gary Moore,Still Got The Blues,spotify:track:0DnGfA1r8pAssJCuq4ojla,clickrow,True,False,...,False,False,False,False,False,False,0.000070,0.000300,0.000110,69.0


In [9]:
# Para procesar los datos de submission, primero se los une con los de entrenamiento sin procesar y se los procesa juntos para
# obtener is_early_finish y racha_skips_prev. Los demás atributos se obtienen con procesar_df_2

# Necesitamos los datos de entrenamiento sin procesar.
data = pd.read_csv('../competition_data.csv')

#necesitamos que submission tenga una columna de TARGET, aunque sea NaN, para poder concatenar
submission['TARGET'] = np.nan
submission = func.procesar_df_2(submission, diccionario)

#unimos los datos de submission con los de entrenamiento
union = pd.concat([data, submission.copy()], axis=0)

# Procesamos la unión de los datos de entrenamiento y de submission
union = func.procesar_df(union, diccionario)

# Ahora unimos submission con las columnas de union que nos interesan
submission = submission.merge(
    union[['Unnamed: 0', 'is_early_finish', 'racha_skips_prev']],
    on='Unnamed: 0',
    how='left'
)

# Nos aseguramos de que las columnas de submission estén en el orden correcto para el modelo
submission = submission[union.columns]

submission

,Unnamed: 0,ts,platform,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,reason_start,shuffle,TARGET,...,reason_appload,reason_backbtn,reason_clickrow,reason_fwdbtn,reason_playbtn,reason_remote,reason_trackdone,reason_trackerror,reason_unknown,hour day_of_week
0,74916,2014-06-27 18:01:15+00:00,"iOS 7.0.4 (iPod5,1)",Mejor,Los Tipitos,Push,spotify:track:5LFl6vXC2CwcciAbymL4jZ,clickrow,False,NaN,...,False,False,True,False,False,False,False,False,False,72.0
1,74923,2014-09-04 21:46:57+00:00,"iOS 7.0.4 (iPod5,1)","Circles - Based On Ludovico Einaudi ""Experience""",Ludovico Einaudi,In a Time Lapse,spotify:track:0mEsOEi4rWBy0IXE5oTKr2,fwdbtn,False,NaN,...,False,False,False,True,False,False,False,False,False,63.0
2,74924,2014-09-04 21:48:51+00:00,"iOS 7.0.4 (iPod5,1)",Primavera,Ludovico Einaudi,Divenire,spotify:track:0fzw4BBD5FRJtPuQbUUKzJ,NaN,False,NaN,...,False,False,False,False,False,False,False,False,False,63.0
3,74933,2016-06-23 21:07:59+00:00,OS X 10.11.5 [x86 4],NaN,NaN,NaN,NaN,clickrow,False,NaN,...,False,False,True,False,False,False,False,False,False,63.0
4,74934,2016-06-23 21:08:03+00:00,OS X 10.11.5 [x86 4],NaN,NaN,NaN,NaN,clickrow,False,NaN,...,False,False,True,False,False,False,False,False,False,63.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25032,74898,2024-05-22 15:28:48+00:00,ios,Zafar,La Vela Puerca,A Contraluz,spotify:track:1wIUWGdTdhVk5gIPd0ULxX,trackdone,True,NaN,...,False,False,False,False,False,False,True,False,False,30.0
25033,74899,2024-05-22 15:35:04+00:00,ios,Un Loco En La Calesita,Juan Carlos Baglietto,Baglietto,spotify:track:3mHOEGxXbUpk5CZDgQhrUP,fwdbtn,True,NaN,...,False,False,False,True,False,False,False,False,False,30.0
25034,74900,2024-05-22 15:39:44+00:00,ios,Dulce condena - Edición Aniversario,Los Rodriguez,Sin Documentos,spotify:track:4Pk1N5mY14kO5N3JcADgb2,trackdone,True,NaN,...,False,False,False,False,False,False,True,False,False,30.0
25035,74901,2024-05-22 15:39:49+00:00,ios,Yo No Quiero Volverme Tan Loco,Charly García,Pubis Angelical / Yendo De La Cama Al Living,spotify:track:68LeIVjVDRMXPlfdFHhID6,trackdone,True,NaN,...,False,False,False,False,False,False,True,False,False,30.0


### Validación de modelos

#### Primer modelo: holdout set con temporalidad

In [10]:
data = pd.read_csv('../competition_data.csv')

In [11]:
# Ordenamos los datos de entrenamiento por timestamp
data = func.sort_by_ts(data)

# Generación de los conjuntos de entrenamiento y validación
n = len(data)
split_idx = int(n * 0.8)
train = data.iloc[:split_idx].copy()
validation = data.iloc[split_idx:].copy()
target_val = validation['TARGET'].copy()

# Seteamos la columna de TARGET a NaN en el conjunto de validación
validation['TARGET'] = np.nan

In [12]:
#unimos los datos de submission con los de entrenamiento
union = pd.concat([train.copy(), validation.copy()], axis=0)

# Procesamos la unión de los datos de entrenamiento y de validación
union = func.procesar_df(union, diccionario)

In [13]:
# Procesamos los datos de entrenamiento y validacion sin is early_finish y racha_skips_prev
train = func.procesar_df(train, diccionario)
validation = func.procesar_df_2(validation, diccionario)

# Ahora unimos validation con las columnas de union que nos interesan
validation = validation.merge(
    union[['Unnamed: 0', 'is_early_finish', 'racha_skips_prev']],
    on='Unnamed: 0',
    how='left'
)

# Nos aseguramos de que las columnas de submission estén en el orden correcto para el modelo
validation = validation[union.columns]

In [14]:
# Bloque donde se entrena el modelo y se calcula AUC-ROC
import xgboost as xgb
from sklearn.metrics import roc_auc_score

# Entrenamiento del modelo
drop_cols = [
    'Unnamed: 0','ts','TARGET', 'platform', 'reason_start',
    'master_metadata_track_name',
    'master_metadata_album_artist_name',
    'master_metadata_album_album_name',
    'spotify_track_uri','platform'
]
clf_xgb = xgb.XGBClassifier(objective = 'binary:logistic',
                            seed = 42,
                            eval_metric = 'auc',
                            early_stopping_rounds = 100)
clf_xgb.fit(train.drop(columns=drop_cols),
                train['TARGET'],
                eval_set=[(validation.drop(columns=drop_cols), target_val)],
                verbose=True
                )

# Obtención de la mejor iteración y cálculo del AUC-ROC
best = clf_xgb.best_iteration
print(best)
prediccion_val = clf_xgb.predict_proba(validation.drop(columns=drop_cols))[:, 1]
auc = roc_auc_score(target_val, prediccion_val)
print(f"AUC-ROC: {auc:.4f}")

[0]	validation_0-auc:0.86383
[1]	validation_0-auc:0.86623
[2]	validation_0-auc:0.88114
[3]	validation_0-auc:0.88300
[4]	validation_0-auc:0.88427
[5]	validation_0-auc:0.89293
[6]	validation_0-auc:0.90005
[7]	validation_0-auc:0.89974
[8]	validation_0-auc:0.90171
[9]	validation_0-auc:0.90282
[10]	validation_0-auc:0.90447
[11]	validation_0-auc:0.90961
[12]	validation_0-auc:0.90945
[13]	validation_0-auc:0.90906
[14]	validation_0-auc:0.90856
[15]	validation_0-auc:0.90794
[16]	validation_0-auc:0.90836
[17]	validation_0-auc:0.90800
[18]	validation_0-auc:0.90732
[19]	validation_0-auc:0.90666
[20]	validation_0-auc:0.90586
[21]	validation_0-auc:0.90560
[22]	validation_0-auc:0.90466
[23]	validation_0-auc:0.90451
[24]	validation_0-auc:0.90401
[25]	validation_0-auc:0.90388
[26]	validation_0-auc:0.90383
[27]	validation_0-auc:0.90332
[28]	validation_0-auc:0.90273
[29]	validation_0-auc:0.90260
[30]	validation_0-auc:0.90217
[31]	validation_0-auc:0.90133
[32]	validation_0-auc:0.90150
[33]	validation_0-au

#### Segundo modelo: Time Series Cross Validation

In [15]:
data = pd.read_csv('../competition_data.csv')
data = func.sort_by_ts(data)

In [18]:
from sklearn.model_selection import TimeSeriesSplit

n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)


In [19]:
auc_scores = []
for fold, (train_idx, val_idx) in enumerate(tscv.split(data), start=1):
    print(f"Fold {fold}")

    train_fold = data.loc[train_idx].copy().reset_index(drop=True)
    val_fold   = data.loc[val_idx].copy().reset_index(drop=True)
    target_val = val_fold['TARGET'].copy()
    val_fold['TARGET'] = np.nan

    #unimos los datos de submission con los de entrenamiento
    union = pd.concat([train_fold.copy(), val_fold.copy()], axis=0)
    # Procesamos la unión de los datos de entrenamiento y de validación
    union = func.procesar_df(union, diccionario)

    # Procesamos los datos de entrenamiento y validacion sin is early_finish y racha_skips_prev
    train_fold = func.procesar_df(train_fold, diccionario)
    val_fold   = func.procesar_df_2(val_fold,   diccionario)

    val_fold = val_fold.merge(
        union[['Unnamed: 0', 'is_early_finish', 'racha_skips_prev']],
        on='Unnamed: 0',
        how='left'
    )

    val_fold = val_fold[union.columns]

    X_tr = train_fold.drop(columns=drop_cols)
    y_tr = train_fold['TARGET']
    X_va = val_fold  .drop(columns=drop_cols)
    y_va = target_val
    
    clf = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        seed=42,
        early_stopping_rounds=50
    )
    clf.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False
    )

    preds = clf.predict_proba(X_va)[:, 1]
    auc   = roc_auc_score(y_va, preds)
    print(f"   AUC Fold {fold}: {auc:.4f}")
    best = clf.best_iteration
    print(best)
    auc_scores.append(auc)

# Resultados globales
print(f"\n✅ AUC mean: {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")

Fold 1
   AUC Fold 1: 0.7069
0
Fold 2
   AUC Fold 2: 0.9042
24
Fold 3
   AUC Fold 3: 0.9698
13
Fold 4
   AUC Fold 4: 0.9024
26
Fold 5
   AUC Fold 5: 0.9116
20

✅ AUC mean: 0.8790 ± 0.0896


#### Tercer modelo: holdout set aleatorio

In [31]:
# Este último modelo lo implementamos en una función para que sea más fácil de usar
data = pd.read_csv('../competition_data.csv')
data = func.sort_by_ts(data)

In [32]:
# Toma aleatoriamente el 70 % de las filas
train = data.sample(frac=0.7, random_state=42).copy()

# El resto (30 %) lo podés obtener excluyendo esos índices
validation = data.drop(train.index).copy()
target_val = validation['TARGET'].copy()
validation['TARGET'] = np.nan

In [33]:
union = pd.concat([train, validation], axis=0)
union = func.procesar_df(union, diccionario)
train = func.procesar_df(train, diccionario)
validation = func.procesar_df_2(validation, diccionario)

validation = validation.merge(
    union[['Unnamed: 0', 'is_early_finish', 'racha_skips_prev']],
    on='Unnamed: 0',
    how='left'
)

validation = validation[train.columns]

In [34]:
drop_cols = [
    'Unnamed: 0','ts', 'platform', 'reason_start',
    'master_metadata_track_name',
    'master_metadata_album_artist_name',
    'master_metadata_album_album_name',
    'spotify_track_uri','platform', 'TARGET'
]
clf_xgb = xgb.XGBClassifier(objective = 'binary:logistic',
                            seed = 42,
                            eval_metric = 'auc',
                            early_stopping_rounds = 100)
clf_xgb.fit(train.drop(columns=drop_cols),
                train['TARGET'],
                eval_set=[(validation.drop(columns=drop_cols), target_val)],
                verbose=True
                )

[0]	validation_0-auc:0.95763
[1]	validation_0-auc:0.95830
[2]	validation_0-auc:0.95954
[3]	validation_0-auc:0.96069
[4]	validation_0-auc:0.96116
[5]	validation_0-auc:0.96135
[6]	validation_0-auc:0.96183
[7]	validation_0-auc:0.96215
[8]	validation_0-auc:0.96224
[9]	validation_0-auc:0.96236
[10]	validation_0-auc:0.96265
[11]	validation_0-auc:0.96285
[12]	validation_0-auc:0.96308
[13]	validation_0-auc:0.96351
[14]	validation_0-auc:0.96361
[15]	validation_0-auc:0.96376
[16]	validation_0-auc:0.96384
[17]	validation_0-auc:0.96407
[18]	validation_0-auc:0.96413
[19]	validation_0-auc:0.96422
[20]	validation_0-auc:0.96424
[21]	validation_0-auc:0.96480
[22]	validation_0-auc:0.96484
[23]	validation_0-auc:0.96482
[24]	validation_0-auc:0.96481
[25]	validation_0-auc:0.96483
[26]	validation_0-auc:0.96503
[27]	validation_0-auc:0.96501
[28]	validation_0-auc:0.96507
[29]	validation_0-auc:0.96507
[30]	validation_0-auc:0.96505
[31]	validation_0-auc:0.96507
[32]	validation_0-auc:0.96509
[33]	validation_0-au

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=100,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [35]:
best = clf_xgb.best_iteration
print(best)

prediccion_val = clf_xgb.predict_proba(validation.drop(columns=drop_cols))[:, 1]
auc = roc_auc_score(target_val.astype(int), prediccion_val)
print(f"AUC-ROC: {auc:.4f}")

50
AUC-ROC: 0.9653


### Búsqueda de hiperparámetros

In [39]:
drop_cols = [
    'Unnamed: 0','ts', 'platform', 'reason_start',
    'master_metadata_track_name',
    'master_metadata_album_artist_name',
    'master_metadata_album_album_name',
    'spotify_track_uri','platform', 'TARGET'
]

for semilla in [42, 678, 831, 744, 213]:
    print(f"Semilla: {semilla}")
    data = pd.read_csv('../competition_data.csv')
    data = func.sort_by_ts(data)

    # Toma aleatoriamente el 70 % de las filas con la semilla dada
    train = data.sample(frac=0.7, random_state=semilla).copy()

    # El resto (30 %) lo podés obtener excluyendo esos índices
    validation = data.drop(train.index).copy()
    target_val = validation['TARGET'].copy()
    validation['TARGET'] = np.nan

    union = pd.concat([train, validation], axis=0)
    union = func.procesar_df(union, diccionario)
    train = func.procesar_df(train, diccionario)
    validation = func.procesar_df_2(validation, diccionario)

    validation = validation.merge(
        union[['Unnamed: 0', 'is_early_finish', 'racha_skips_prev']],
        on='Unnamed: 0',
        how='left'
    )

    validation = validation[train.columns]

    aucs = {}
    for l in [0.1, 1, 5, 10]:
        for g in [0, 0.1, 0.5, 1]:
            for e in [0.01, 0.05, 0.1, 0.3]:
                clf_xgb = xgb.XGBClassifier(objective = 'binary:logistic',
                                            seed = semilla,
                                            eval_metric = 'auc',
                                            early_stopping_rounds = 100,
                                            learning_rate = e,
                                            gamma = g,
                                            reg_lambda=l)
                clf_xgb.fit(train.drop(columns=drop_cols),
                                train['TARGET'],
                                eval_set=[(validation.drop(columns=drop_cols), target_val)],
                                verbose=False
                                )
                best = clf_xgb.best_iteration
                prediccion_val = clf_xgb.predict_proba(validation.drop(columns=drop_cols))[:, 1]
                auc = roc_auc_score(target_val, prediccion_val)
                aucs[(l, g, e)] = auc

    best = None
    best_auc = 0
    for clave in aucs.keys():
        if best is None or aucs[clave] > aucs[best]:
            best = clave
            best_auc = aucs[clave]

    print(f"Mejor combinación: {best} con AUC-ROC: {best_auc:.4f}")

Semilla: 42
Mejor combinación: (5, 0, 0.3) con AUC-ROC: 0.9657
Semilla: 678
Mejor combinación: (5, 0, 0.3) con AUC-ROC: 0.9650
Semilla: 831
Mejor combinación: (10, 0, 0.3) con AUC-ROC: 0.9641
Semilla: 744
Mejor combinación: (10, 0.5, 0.3) con AUC-ROC: 0.9648
Semilla: 213
Mejor combinación: (5, 0, 0.3) con AUC-ROC: 0.9650
